# Grid-Based Declustering

This notebook demonstrates grid-based declustering using inverse distance weighting.

Before running this notebook, create a Seequent Evo workspace and populate it with the example data using [1-populate-workspace.ipynb](./1-populate-workspace.ipynb).

---

As with any Python-based workflow using Seequent Evo, first import the necessary packages and authenticate to access the data.

In [ ]:
import pandas as pd
import plotting
from dotenv import dotenv_values
from helpers import compare_weighted_summary

import evo.objects.typed as objs
from evo.compute import tasks
from evo.compute.tasks.geostatistics import declustering
from evo.notebooks import ServiceManagerWidget

%load_ext evo.widgets

In [ ]:
evo_connection = await ServiceManagerWidget.with_auth_code(**dotenv_values()).login()

## Load the Objects Needed

When accessing an object from the Evo workspace, a reference to that object is returned. The reference contains information about the unique ID and location of the object, as well as its coordinates and attributes. The object reference does **not** contain the actual numerical data.

The object reference and its numerical data are intentionally separate, so workflows can be orchestrated on Evo without downloading all data locally. To access the underlying numerical data when needed, use the `.to_dataframe` method.

In this case, a pointset and a block model are needed.

In [ ]:
comps = await objs.object_from_path(evo_connection, "WP comps.json")
block_model = await objs.object_from_path(evo_connection, "WP Regular BlockModel.json")

To inspect the reference information, such as metadata and the list of attributes, take advantage of the HTML widget associated with the object.

In [ ]:
comps

## Set Up Declustering Parameters

Each geostatistics compute task accepts inputs and parameters. Details are available in the [developer portal](https://developer.seequent.com/docs/api/geostatistics-task/geostatistics-task-api).

For this example, first prepare parameters for inverse distance weighted (IDW) declustering.

In [ ]:
# first create the search neighborhood
ellipsoid = tasks.Ellipsoid(
    ranges=(300, 150, 100),
    rotation=(70, 15, 0),
)
srch = tasks.SearchNeighborhood(ellipsoid=ellipsoid, min_samples=1, max_samples=10)


# next, establish the rest of the parameters
declus_params = declustering.DeclusteringParameters(
    source=comps,
    grid=block_model,
    target=comps.attributes["IDW declus"],
    neighborhood=srch,
    power=2.0,
)

## Execute the Compute Task

Once the parameters are established, execute the task using the `.run` method. The `await` keyword is required here to run the task in a non-blocking the way, allowing other asynchronous work to proceed. To run multiple tasks concurrently, submit them together as a list: `[task_params_1, task_params_2]`.

In [ ]:
declus_result = await tasks.run(evo_connection, declus_params, preview = True)

## Investigate Results

Because compute tasks are orchestrated by the user and executed on Evo, the results are saved directly to the Seequent Evo workspace. To investigate them locally, reload the object reference from the workspace.

In [ ]:
updated_comps = await comps.refresh()
updated_comps

The newly created declustering weights can be viewed in the Seequent Evo Viewer by clicking the `Viewer` link above, or downloaded locally for statistical evaluation.

To download them locally, use the `.to_dataframe` method and specify only the necessary columns.

In [ ]:
comps_df = await updated_comps.to_dataframe("Cu_pct", "IDW declus")
comps_df.head()

Using [Plotly](https://docs.plotly.com/), generate a histogram comparing the original and IDW-declustered `Cu_pct` distributions.

In [ ]:
comparison = pd.concat(
    [
        comps_df.assign(distribution="Original", weight=1.0),
        comps_df.assign(
            distribution="IDW declustered",
            weight=comps_df["IDW declus"],
        ),
    ],
    ignore_index=True,
)

hist = plotting.histogram_comparison(
    comparison,
    x="Cu_pct",
    color="distribution",
    weights="weight",
    category_order=["Original", "IDW declustered"],
    colors=["steelblue", "darkorange"],
    title="Cu_pct Distribution: Original vs IDW Declustered",
)
hist.show()

Now examine the weighted statistics. [pandas' `.describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) does not provide weighted statistics, instead [NumPy's `.average`](https://numpy.org/doc/stable/reference/generated/numpy.average.html) method for this analysis.

In [ ]:
compare_weighted_summary(values=comps_df["Cu_pct"], weights=comps_df["IDW declus"])